In [3]:
"""
Spam or Ham: Model Training & Saving
------------------------------------
This script:
1) Downloads the Kaggle dataset ("purusinghvi/email-spam-classification-dataset") via kagglehub
2) Preprocesses text (lowercase, clean, tokenize, stopword removal, stemming)
3) Implements Multinomial Naive Bayes from scratch with Laplace smoothing
4) Splits data, tunes alpha, evaluates with from-scratch metrics
5) Saves the trained model and fitted CountVectorizer to disk

Requirements:
- kagglehub, pandas, numpy, nltk, scikit-learn
- NLTK resources 'punkt' and 'stopwords' (downloaded automatically if missing)
"""

import os
import re
import pickle
import warnings
import numpy as np
import pandas as pd

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

# ML utils
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

# Data source
import kagglehub

warnings.filterwarnings("ignore")



In [9]:

# ---------------------------
# NLTK setup & preprocessing
# ---------------------------
def ensure_nltk():
    """Download required NLTK assets if missing."""
    try:
        nltk.data.find("tokenizers/punkt")
    except LookupError:
        nltk.download("punkt", quiet=True)
    try:
        nltk.data.find("corpora/stopwords")
    except LookupError:
        nltk.download("stopwords", quiet=True)


def preprocess_text_factory():
    """Prepare stopwords and stemmer; return a preprocess_text function bound to them."""
    stop_words = set(stopwords.words("english"))
    stemmer = PorterStemmer()

    def preprocess_text(text: str) -> str:
        """Cleans, tokenizes, removes stopwords, and stems text."""
        if not isinstance(text, str):
            return ""
        # Lowercase
        text_l = text.lower()
        # Remove URLs, emails, numbers, punctuation
        text_l = re.sub(r"http\S+|www\S+|https\S+", "", text_l, flags=re.MULTILINE)
        text_l = re.sub(r"\S*@\S*\s?", "", text_l)
        text_l = re.sub(r"\d+", "", text_l)
        text_l = re.sub(r"[^a-z\s]", "", text_l)
        # Tokenize
        tokens = word_tokenize(text_l)
        # Remove stopwords & stem; keep tokens length > 2
        processed = [stemmer.stem(w) for w in tokens if w not in stop_words and len(w) > 2]
        return " ".join(processed)

    return preprocess_text


In [5]:


# ---------------------------
# Multinomial Naive Bayes (from scratch)
# ---------------------------
class MultinomialNB_from_scratch:
    def __init__(self, alpha: float = 1.0):
        """Laplace smoothing parameter alpha >= 0."""
        self.alpha = float(alpha)
        self.log_prior_spam = 0.0
        self.log_prior_ham = 0.0
        self.log_likelihood_spam = None  # shape: (n_features,)
        self.log_likelihood_ham = None  # shape: (n_features,)

    def fit(self, X, y):
        """
        X: scipy.sparse matrix (n_samples, n_features) of counts
        y: array-like (n_samples,) with labels {0: ham, 1: spam}
        """
        y = np.asarray(y)
        n_samples, n_features = X.shape

        # Priors
        n_spam = np.sum(y == 1)
        n_ham = n_samples - n_spam
        self.log_prior_spam = np.log((n_spam + 1e-12) / (n_samples + 1e-12))
        self.log_prior_ham = np.log((n_ham + 1e-12) / (n_samples + 1e-12))

        # Split rows by class
        X_spam = X[y == 1]
        X_ham = X[y == 0]

        # Word counts + Laplace smoothing
        spam_word_counts = np.asarray(X_spam.sum(axis=0)).ravel() + self.alpha
        ham_word_counts = np.asarray(X_ham.sum(axis=0)).ravel() + self.alpha

        # Totals (denominator includes alpha*V because we added alpha per feature)
        total_spam_words = spam_word_counts.sum()
        total_ham_words = ham_word_counts.sum()

        # Log-likelihoods
        self.log_likelihood_spam = np.log(spam_word_counts / total_spam_words)
        self.log_likelihood_ham = np.log(ham_word_counts / total_ham_words)

        return self

    def predict(self, X):
        """
        X: scipy.sparse matrix (n_samples, n_features)
        returns: np.array of predictions {0, 1}
        """
        log_scores_spam = X @ self.log_likelihood_spam + self.log_prior_spam
        log_scores_ham = X @ self.log_likelihood_ham + self.log_prior_ham
        preds = (log_scores_spam > log_scores_ham).astype(int)
        return np.asarray(preds)



In [6]:

# ---------------------------
# Metrics (from scratch)
# ---------------------------
def print_metrics(y_true, y_pred):
    """
    Calculates and prints accuracy, precision, recall, F1, and confusion matrix.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    accuracy = np.mean(y_pred == y_true)
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    tn = np.sum((y_pred == 0) & (y_true == 0))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    print(f"--- Evaluation Metrics (Spam = 1) ---")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print("\n--- Confusion Matrix ---")
    print(f"\t\tPredicted Ham\tPredicted Spam")
    print(f"Actual Ham\t{tn}\t\t{fp}")
    print(f"Actual Spam\t{fn}\t\t{tp}")




In [7]:
# ---------------------------
# Main training & saving flow
# ---------------------------
def main():
    print("=== Spam/Ham: Model Training & Saving ===")

    # 0) Download dataset using kagglehub
    print("\n[Step 0] Downloading dataset via kagglehub...")
    path = kagglehub.dataset_download("purusinghvi/email-spam-classification-dataset")
    print("Path to dataset files:", path)

    # 1) Setup NLTK
    print("\n[Step 1] Ensuring NLTK resources...")
    ensure_nltk()
    preprocess_text = preprocess_text_factory()
    print("NLTK ready.")

    # 2) Load data
    print("\n[Step 2] Loading dataset CSV...")
    file_path = os.path.join(path, "combined_spam_data.csv")
    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"Error: Could not find the file at {file_path}. "
            "Please ensure the dataset downloaded correctly."
        )

    df = pd.read_csv(file_path)
    print("Dataset loaded successfully.")
    print(f"Total emails: {len(df)}")

    print("\n--- First 5 Rows ---")
    with pd.option_context("display.max_colwidth", 80):
        print(df.head())

    print("\n--- Data Info (summary dtypes) ---")
    print(df.dtypes)

    # 3) Data Preprocessing & Label Encoding
    print("\n[Step 3] Preprocessing & label encoding...")
    if "label" not in df.columns or "text" not in df.columns:
        raise ValueError("Expected columns 'label' and 'text' not found in CSV.")
    df["label"] = df["label"].map({"spam": 1, "ham": 0})
    df.dropna(subset=["label", "text"], inplace=True)
    df["label"] = df["label"].astype(int)

    print("\n--- Label Distribution (normalized) ---")
    print(df["label"].value_counts(normalize=True))

    print("Starting text preprocessing...")
    df["processed_text"] = df["text"].apply(preprocess_text)
    print("Text preprocessing complete.")

    print("\n--- Processed Data Example ---")
    print(df[["text", "processed_text", "label"]].head())

    # 4) Split Data and Vectorize
    print("\n[Step 4] Train/test split & vectorization...")
    X = df["processed_text"]
    y = df["label"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    print(f"Training samples: {len(X_train)}")
    print(f"Test samples:     {len(X_test)}")

    vectorizer = CountVectorizer(tokenizer=lambda x: x.split())
    X_train_bow = vectorizer.fit_transform(X_train)
    X_test_bow = vectorizer.transform(X_test)

    print("\nData vectorized.")
    print(f"Vocabulary size: {len(vectorizer.get_feature_names_out())}")
    print(f"Shape of training matrix: {X_train_bow.shape}")

    # 5) MultinomialNB_from_scratch defined above

    # 6) Hyperparameter Tuning
    print("\n[Step 6] Hyperparameter tuning (alpha)...")
    alphas_to_try = [0.1, 0.5, 1.0, 2.0, 5.0]
    best_alpha = None
    best_accuracy = -1.0

    for alpha in alphas_to_try:
        model = MultinomialNB_from_scratch(alpha=alpha)
        model.fit(X_train_bow, y_train)
        y_pred = model.predict(X_test_bow)
        accuracy = float(np.mean(y_pred == y_test))
        print(f"Alpha = {alpha:.1f} | Accuracy = {accuracy:.4f}")
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_alpha = alpha

    print(f"\nBest Alpha: {best_alpha}")
    print(f"Best Test Accuracy: {best_accuracy:.4f}")

    # 7) Final Training & Evaluation
    print("\n[Step 7] Training final model & evaluation...")
    final_model = MultinomialNB_from_scratch(alpha=best_alpha)
    final_model.fit(X_train_bow, y_train)
    y_final_pred = final_model.predict(X_test_bow)
    print_metrics(y_test, y_final_pred)

    # 8) Save the Trained Model and Vectorizer
    print("\n[Step 8] Saving model and vectorizer...")
    model_filename = "spam_classifier.pkl"
    vectorizer_filename = "vectorizer.pkl"

    with open(model_filename, "wb") as f:
        pickle.dump(final_model, f)

    with open(vectorizer_filename, "wb") as f:
        pickle.dump(vectorizer, f)

    print(f"Model saved to: {model_filename}")
    print(f"Vectorizer saved to: {vectorizer_filename}")

    print("\n--- Example: How to load them later ---")
    print("\n# Load the vectorizer")
    print("with open('vectorizer.pkl', 'rb') as f:")
    print("    loaded_vectorizer = pickle.load(f)")
    print("\n# Load the model")
    print("with open('spam_classifier.pkl', 'rb') as f:")
    print("    loaded_model = pickle.load(f)")

    print("\n=== Done. ===")


In [8]:
main()


=== Spam/Ham: Model Training & Saving ===

[Step 0] Downloading dataset via kagglehub...
Path to dataset files: C:\Users\shiva\.cache\kagglehub\datasets\purusinghvi\email-spam-classification-dataset\versions\1

[Step 1] Ensuring NLTK resources...
NLTK ready.

[Step 2] Loading dataset CSV...


FileNotFoundError: Error: Could not find the file at C:\Users\shiva\.cache\kagglehub\datasets\purusinghvi\email-spam-classification-dataset\versions\1\combined_spam_data.csv. Please ensure the dataset downloaded correctly.

In [12]:
def ensure_nltk():
    """Download required NLTK assets if missing."""
    # This is more robust. NLTK is smart and won't re-download
    # if the packages are already present and correct.
    nltk.download("punkt", quiet=True)
    nltk.download("stopwords", quiet=True)

In [13]:
"""
Spam or Ham: Model Training & Saving
------------------------------------
This script:
1) Downloads the Kaggle dataset ("purusinghvi/email-spam-classification-dataset") via kagglehub
2) Preprocesses text (lowercase, clean, tokenize, stopword removal, stemming)
3) Implements Multinomial Naive Bayes from scratch with Laplace smoothing
4) Splits data, tunes alpha, evaluates with from-scratch metrics
5) Saves the trained model and fitted CountVectorizer to disk

Requirements:
- kagglehub, pandas, numpy, nltk, scikit-learn
- NLTK resources 'punkt' and 'stopwords' (downloaded automatically if missing)
"""

import os
import re
import pickle
import warnings
import numpy as np
import pandas as pd

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

# ML utils
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

# Data source
import kagglehub

warnings.filterwarnings("ignore")


# ---------------------------
# NLTK setup & preprocessing
# ---------------------------
def ensure_nltk():
    """Download required NLTK assets if missing."""
    # --- THIS IS THE FIX ---
    # This is more robust. NLTK is smart and won't re-download
    # if the packages are already present and correct.
    nltk.download("punkt", quiet=True)
    nltk.download("stopwords", quiet=True)
    # --- END FIX ---


def preprocess_text_factory():
    """Prepare stopwords and stemmer; return a preprocess_text function bound to them."""
    stop_words = set(stopwords.words("english"))
    stemmer = PorterStemmer()

    def preprocess_text(text: str) -> str:
        """Cleans, tokenizes, removes stopwords, and stems text."""
        if not isinstance(text, str):
            return ""
        # Lowercase
        text_l = text.lower()
        # Remove URLs, emails, numbers, punctuation
        text_l = re.sub(r"http\S+|www\S+|https\S+", "", text_l, flags=re.MULTILINE)
        text_l = re.sub(r"\S*@\S*\s?", "", text_l)
        text_l = re.sub(r"\d+", "", text_l)
        text_l = re.sub(r"[^a-z\s]", "", text_l)
        # Tokenize
        tokens = word_tokenize(text_l)
        # Remove stopwords & stem; keep tokens length > 2
        processed = [stemmer.stem(w) for w in tokens if w not in stop_words and len(w) > 2]
        return " ".join(processed)

    return preprocess_text


# ---------------------------
# Multinomial Naive Bayes (from scratch)
# ---------------------------
class MultinomialNB_from_scratch:
    def __init__(self, alpha: float = 1.0):
        """Laplace smoothing parameter alpha >= 0."""
        self.alpha = float(alpha)
        self.log_prior_spam = 0.0
        self.log_prior_ham = 0.0
        self.log_likelihood_spam = None  # shape: (n_features,)
        self.log_likelihood_ham = None  # shape: (n_features,)

    def fit(self, X, y):
        """
        X: scipy.sparse matrix (n_samples, n_features) of counts
        y: array-like (n_samples,) with labels {0: ham, 1: spam}
        """
        y = np.asarray(y)
        n_samples, n_features = X.shape

        # Priors
        n_spam = np.sum(y == 1)
        n_ham = n_samples - n_spam
        self.log_prior_spam = np.log((n_spam + 1e-12) / (n_samples + 1e-12))
        self.log_prior_ham = np.log((n_ham + 1e-12) / (n_samples + 1e-12))

        # Split rows by class
        X_spam = X[y == 1]
        X_ham = X[y == 0]

        # Word counts + Laplace smoothing
        spam_word_counts = np.asarray(X_spam.sum(axis=0)).ravel() + self.alpha
        ham_word_counts = np.asarray(X_ham.sum(axis=0)).ravel() + self.alpha

        # Totals (denominator includes alpha*V because we added alpha per feature)
        total_spam_words = spam_word_counts.sum()
        total_ham_words = ham_word_counts.sum()

        # Log-likelihoods
        self.log_likelihood_spam = np.log(spam_word_counts / total_spam_words)
        self.log_likelihood_ham = np.log(ham_word_counts / total_ham_words)

        return self

    def predict(self, X):
        """
        X: scipy.sparse matrix (n_samples, n_features)
        returns: np.array of predictions {0, 1}
        """
        log_scores_spam = X @ self.log_likelihood_spam + self.log_prior_spam
        log_scores_ham = X @ self.log_likelihood_ham + self.log_prior_ham
        preds = (log_scores_spam > log_scores_ham).astype(int)
        return np.asarray(preds)


# ---------------------------
# Metrics (from scratch)
# ---------------------------
def print_metrics(y_true, y_pred):
    """
    Calculates and prints accuracy, precision, recall, F1, and confusion matrix.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    accuracy = np.mean(y_pred == y_true)
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    tn = np.sum((y_pred == 0) & (y_true == 0))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    print(f"--- Evaluation Metrics (Spam = 1) ---")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print("\n--- Confusion Matrix ---")
    print(f"\t\tPredicted Ham\tPredicted Spam")
    print(f"Actual Ham\t{tn}\t\t{fp}")
    print(f"Actual Spam\t{fn}\t\t{tp}")


# ---------------------------
# Main training & saving flow
# ---------------------------
def main():
    print("=== Spam/Ham: Model Training & Saving ===")

    # 0) Download dataset using kagglehub
    print("\n[Step 0] Downloading dataset via kagglehub...")
    path = kagglehub.dataset_download("purusinghvi/email-spam-classification-dataset")
    print("Path to dataset files:", path)

    # 1) Setup NLTK
    print("\n[Step 1] Ensuring NLTK resources...")
    ensure_nltk()
    preprocess_text = preprocess_text_factory()
    print("NLTK ready.")

    # 2) Load data
    print("\n[Step 2] Loading dataset CSV...")

    # Scan the directory 'path' for the first file ending in .csv
    file_path = None
    for file in os.listdir(path):
        if file.endswith(".csv"):
            file_path = os.path.join(path, file)
            break  # Found it

    # If not found in the root, check one level deeper
    if file_path is None:
        for dir_item in os.listdir(path):
            potential_subdir = os.path.join(path, dir_item)
            if os.path.isdir(potential_subdir):
                for file in os.listdir(potential_subdir):
                    if file.endswith(".csv"):
                        file_path = os.path.join(potential_subdir, file)
                        break
            if file_path:
                break  # Found in a subdir

    if file_path is None or not os.path.exists(file_path):
        raise FileNotFoundError(
            f"Error: Could not find a .csv file in {path} or its subdirectories. "
            "Please ensure the dataset downloaded and extracted correctly."
        )

    print(f"Loading from located file: {file_path}")

    df = pd.read_csv(file_path)
    print("Dataset loaded successfully.")
    print(f"Total emails: {len(df)}")

    print("\n--- First 5 Rows ---")
    with pd.option_context("display.max_colwidth", 80):
        print(df.head())

    print("\n--- Data Info (summary dtypes) ---")
    print(df.dtypes)

    # 3) Data Preprocessing & Label Encoding
    print("\n[Step 3] Preprocessing & label encoding...")
    if "label" not in df.columns or "text" not in df.columns:
        raise ValueError("Expected columns 'label' and 'text' not found in CSV.")
    
    # The 'label' column is already 1s and 0s
    # We just drop any rows that might have missing labels or text
    df.dropna(subset=["label", "text"], inplace=True) 
    
    # Ensure the label column is integer type (this is good practice)
    df["label"] = df["label"].astype(int)

    print("\n--- Label Distribution (normalized) ---")
    print(df["label"].value_counts(normalize=True))

    print("Starting text preprocessing...")
    df["processed_text"] = df["text"].apply(preprocess_text)
    print("Text preprocessing complete.")

    print("\n--- Processed Data Example ---")
    print(df[["text", "processed_text", "label"]].head())

    # 4) Split Data and Vectorize
    print("\n[Step 4] Train/test split & vectorization...")
    X = df["processed_text"]
    y = df["label"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    print(f"Training samples: {len(X_train)}")
    print(f"Test samples:     {len(X_test)}")

    vectorizer = CountVectorizer(tokenizer=lambda x: x.split())
    X_train_bow = vectorizer.fit_transform(X_train)
    X_test_bow = vectorizer.transform(X_test)

    print("\nData vectorized.")
    print(f"Vocabulary size: {len(vectorizer.get_feature_names_out())}")
    print(f"Shape of training matrix: {X_train_bow.shape}")

    # 5) MultinomialNB_from_scratch defined above

    # 6) Hyperparameter Tuning
    print("\n[Step 6] Hyperparameter tuning (alpha)...")
    alphas_to_try = [0.1, 0.5, 1.0, 2.0, 5.0]
    best_alpha = None
    best_accuracy = -1.0

    for alpha in alphas_to_try:
        model = MultinomialNB_from_scratch(alpha=alpha)
        model.fit(X_train_bow, y_train)
        y_pred = model.predict(X_test_bow)
        accuracy = float(np.mean(y_pred == y_test))
        print(f"Alpha = {alpha:.1f} | Accuracy = {accuracy:.4f}")
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_alpha = alpha

    print(f"\nBest Alpha: {best_alpha}")
    print(f"Best Test Accuracy: {best_accuracy:.4f}")

    # 7) Final Training & Evaluation
    print("\n[Step 7] Training final model & evaluation...")
    final_model = MultinomialNB_from_scratch(alpha=best_alpha)
    final_model.fit(X_train_bow, y_train)
    y_final_pred = final_model.predict(X_test_bow)
    print_metrics(y_test, y_final_pred)

    # 8) Save the Trained Model and Vectorizer
    print("\n[Step 8] Saving model and vectorizer...")
    model_filename = "spam_classifier.pkl"
    vectorizer_filename = "vectorizer.pkl"

    with open(model_filename, "wb") as f:
        pickle.dump(final_model, f)

    with open(vectorizer_filename, "wb") as f:
        pickle.dump(vectorizer, f)

    print(f"Model saved to: {model_filename}")
    print(f"Vectorizer saved to: {vectorizer_filename}")

    print("\n--- Example: How to load them later ---")
    print("\n# Load the vectorizer")
    print("with open('vectorizer.pkl', 'rb') as f:")
    print("    loaded_vectorizer = pickle.load(f)")
    print("\n# Load the model")
    print("with open('spam_classifier.pkl', 'rb') as f:")
    print("    loaded_model = pickle.load(f)")

    print("\n=== Done. ===")


if __name__ == "__main__":
    main()

=== Spam/Ham: Model Training & Saving ===

[Step 0] Downloading dataset via kagglehub...
Path to dataset files: C:\Users\shiva\.cache\kagglehub\datasets\purusinghvi\email-spam-classification-dataset\versions\1

[Step 1] Ensuring NLTK resources...
NLTK ready.

[Step 2] Loading dataset CSV...
Loading from located file: C:\Users\shiva\.cache\kagglehub\datasets\purusinghvi\email-spam-classification-dataset\versions\1\combined_data.csv
Dataset loaded successfully.
Total emails: 83448

--- First 5 Rows ---
   label  \
0      1   
1      1   
2      0   
3      1   
4      0   

                                                                              text  
0  ounce feather bowl hummingbird opec moment alabaster valkyrie dyad bread fla...  
1  wulvob get your medircations online qnb ikud viagra escapenumber escapenumbe...  
2   computer connection from cnn com wednesday escapenumber may escapenumber es...  
3  university degree obtain a prosperous future money earning power and the pre..

LookupError: 
**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - 'C:\\Users\\shiva/nltk_data'
    - 'c:\\Users\\shiva\\OneDrive\\Desktop\\DA5401_Big_Data_Lab\\.venv\\nltk_data'
    - 'c:\\Users\\shiva\\OneDrive\\Desktop\\DA5401_Big_Data_Lab\\.venv\\share\\nltk_data'
    - 'c:\\Users\\shiva\\OneDrive\\Desktop\\DA5401_Big_Data_Lab\\.venv\\lib\\nltk_data'
    - 'C:\\Users\\shiva\\AppData\\Roaming\\nltk_data'
    - 'C:\\nltk_data'
    - 'D:\\nltk_data'
    - 'E:\\nltk_data'
**********************************************************************
